# Sprint 2 Campus — Protect the FieldCare service

Continue the same FieldCare project from Sprint 1. This notebook supports C08–C13: caller authentication, secret handling, rate limits, and the secured-service checkpoint.

The provider key stays in the runtime; temporary caller keys are generated in memory. An admitted supported request makes a real OpenRouter call. Invalid, unauthenticated, and limited requests stop before model work. Initial Chroma indexing and admitted generation may use provider quota.

**Colab runtime:** `ServiceProcess` runs Uvicorn in a managed child process bound to loopback, checks readiness with a timeout, and provides an explicit stop. Do not run `uvicorn.run()` in a cell or expose the service through ngrok; a foreground server call would prevent that cell from returning. HTTP request cells wait for their bounded client timeout while leaving the notebook runtime available for subsequent cells.


## Campus lessons in Notion

- [API keys in plain language](https://app.notion.com/p/3ea5a55972ad81e898f1ca03d801e980?pvs=204)
- [Adding API key auth to the scaffold](https://app.notion.com/p/3ea5a55972ad81d3951cecd6d8dbd5c7?pvs=204)
- [What is a secret and where it should live](https://app.notion.com/p/3ea5a55972ad81d48e28c56f84342984?pvs=204)
- [Why AI services need rate limits](https://app.notion.com/p/3ea5a55972ad8157a8d4f24c0a8674e2?pvs=204)
- [Applying per-key rate limits](https://app.notion.com/p/3ea5a55972ad81ac920cf6a901d6388c?pvs=204)
- [Secured service checkpoint](https://app.notion.com/p/3ea5a55972ad8177b84fdac0b397fa8d?pvs=204)


## 1. Set up the cumulative project

Run once in a fresh Colab runtime. The bootstrap loads Module B shared code and copies the FieldCare app or imports your Sprint 1 Campus source checkpoint. Save a separate Drive copy of this notebook.

In [ ]:
import os
from pathlib import Path
import shutil
import subprocess
import sys
from zipfile import ZipFile

REPOSITORY = "https://github.com/richhiey/ai-app-dev_Mod-B.git"
REPO = Path("/content/ai-app-dev_Mod-B")
PROJECT = Path("/content/fieldcare-sprint-2-campus")
SPRINT1_CHECKPOINT = ""  # Optional uploaded Sprint 1 Campus source checkpoint.

if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", REPOSITORY, str(REPO)], check=True)
else:
    # Colab runtimes persist /content; refresh an existing clone before reinstalling it.
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "origin", "main"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO), "-c", str(REPO / "requirements.lock")], check=True)
sys.path.insert(0, str(REPO / "src"))  # Make the editable package visible to this running kernel.

from module_b.openrouter import DEFAULT_CHAT_MODEL, require_openrouter_key
require_openrouter_key(prompt=True)
os.environ.setdefault("OPENROUTER_MODEL", DEFAULT_CHAT_MODEL)

if SPRINT1_CHECKPOINT:
    PROJECT.mkdir(parents=True, exist_ok=True)
    with ZipFile(SPRINT1_CHECKPOINT) as archive:
        archive.extractall(PROJECT)
elif not PROJECT.exists():
    shutil.copytree(REPO / "examples" / "fieldcare", PROJECT)
print("Module B source:", REPO)
print("Sprint 2 workspace:", PROJECT)

## 2. Keep caller and provider credentials separate

The OpenRouter key belongs to the FieldCare service process. A caller key belongs to the application sending an HTTP request. They are different values with different jobs. Generate temporary caller keys without displaying them.

In [ ]:
import os
import secrets
import httpx
from module_b.runtime import ServiceProcess

CALLERS = {
    "dispatch": "FIELDCARE_DISPATCH_KEY",
    "partner": "FIELDCARE_PARTNER_KEY",
}
caller_keys = {label: secrets.token_urlsafe(32) for label in CALLERS}
for label, variable in CALLERS.items():
    os.environ[variable] = caller_keys[label]
print("Temporary caller keys are loaded in memory; their values are not displayed.")

## 3. Attach the caller guard before the handler

SecurityMiddleware authenticates requests before FastAPI validates the body or enters the route. Attach it after all route registrations so the inventory covers every protected POST path. This edit preserves the Sprint 1 routes and response contract.

In [ ]:
settings = """from module_b.security import LimitPolicy

CALLER_ENV = {
    "dispatch": "FIELDCARE_DISPATCH_KEY",
    "partner": "FIELDCARE_PARTNER_KEY",
}
POLICY = None
"""
(PROJECT / "app" / "security_settings.py").write_text(settings)

main_path = PROJECT / "app" / "main.py"
main_source = main_path.read_text()
registration = """
from module_b.security import SecurityMiddleware, protected_post_paths
from app.security_settings import CALLER_ENV, POLICY
app.add_middleware(
    SecurityMiddleware,
    caller_env=CALLER_ENV,
    protected_paths=protected_post_paths(app),
    policy=POLICY,
)
"""
if "SecurityMiddleware" not in main_source:
    main_path.write_text(main_source + registration)

def start_service():
    service = ServiceProcess(
        project_dir=PROJECT,
        env={variable: caller_keys[label] for label, variable in CALLERS.items()},
    )
    service.start()
    return service

service = start_service()

## 4. Compare actual HTTP outcomes

Predict each result before running the cell. Only the recognized, supported request reaches Chroma, LangGraph, and OpenRouter. The final request has a valid caller but an invalid body, so authentication succeeds and FastAPI then returns 422 before retrieval or generation.

In [ ]:
payload = {
    "question": "Which filter and airflow checks are documented?",
    "equipment_id": "EQ-FC-1002",
}
with httpx.Client(base_url=service.base_url, timeout=90) as client:
    cases = [
        ("missing caller key", client.post("/v1/diagnose", json=payload)),
        ("wrong caller key", client.post(
            "/v1/diagnose", json=payload, headers={"X-API-Key": "not-a-valid-key"}
        )),
        ("recognized caller", client.post(
            "/v1/diagnose", json=payload,
            headers={"X-API-Key": caller_keys["dispatch"]}
        )),
        ("recognized caller, invalid body", client.post(
            "/v1/diagnose", json={},
            headers={"X-API-Key": caller_keys["dispatch"]}
        )),
    ]

for label, response in cases:
    body = response.json()
    print(label, "->", response.status_code, body.get("status", body.get("detail")))
service.stop()

### Explain what the statuses establish

- 401 shows that a missing or wrong caller key was rejected at the entry boundary.
- 422 shows that an authenticated request can still fail the input schema.
- 200 with ready shows that an admitted request completed for this run. Review the actual answer against its citations; status does not certify answer quality.
- A valid question missing optional equipment context receives deterministic clarification and does not need model generation.

Use the middleware placement and actual HTTP outcomes as evidence. Do not use a counter from a separate request as a measurement of these calls.

## 5. Apply a per-caller allowance

Choose an allowance for a short burst and keep the 60-second window. The fixed-window limiter counts authenticated requests separately for each caller across protected routes. It is held in one process, so restarting starts new windows.

In [ ]:
PRACTICE_ALLOWANCE = 1  # Change this and predict the result before sending requests.
practice_settings = settings.replace(
    "POLICY = None",
    f"POLICY = LimitPolicy(allowance={PRACTICE_ALLOWANCE}, window_seconds=60)",
)
(PROJECT / "app" / "security_settings.py").write_text(practice_settings)
service = start_service()

Write an ordered request plan before running this cell. Keep the number of supported admitted requests small: each makes a real provider call. Invalid and over-limit requests stop before model generation. The plan below uses a supported route; if your Sprint 1 service added another route, you may include it.

In [ ]:
# Start with rejected callers, then use valid-schema and invalid-schema requests.
# Authenticated invalid bodies count toward the window but stop before generation.
REQUEST_PLAN = [
    {"label": "missing key", "caller": None, "path": "/v1/diagnose", "body": payload},
    {"label": "wrong key", "caller": "wrong", "path": "/v1/diagnose", "body": payload},
]
REQUEST_PLAN.extend(
    {"label": "authenticated invalid body", "caller": "dispatch",
     "path": "/v1/diagnose", "body": {}}
    for _ in range(max(0, PRACTICE_ALLOWANCE - 1))
)
REQUEST_PLAN.extend([
    {"label": "Dispatch supported request", "caller": "dispatch",
     "path": "/v1/diagnose", "body": payload},
    {"label": "Dispatch next request", "caller": "dispatch",
     "path": "/v1/diagnose", "body": payload},
    {"label": "Partner independent allowance", "caller": "partner",
     "path": "/v1/diagnose", "body": payload},
])

practice_responses = []
with httpx.Client(base_url=service.base_url, timeout=90) as client:
    for step in REQUEST_PLAN:
        headers = {}
        if step["caller"] in caller_keys:
            headers["X-API-Key"] = caller_keys[step["caller"]]
        elif step["caller"] == "wrong":
            headers["X-API-Key"] = "not-a-valid-key"
        response = client.post(
            step["path"], json=step["body"], headers=headers,
        )
        practice_responses.append(response)
        body = response.json()
        print(
            step["label"], "->", response.status_code,
            body.get("status", body.get("detail")),
            "Retry-After:", response.headers.get("retry-after"),
        )
service.stop()

## 6. Checkpoint assessment: secure a weekend partner pilot

Set a two-request allowance per caller and design an ordered sequence for the Dispatch and Partner callers. Demonstrate missing and incorrect credentials, a supported admitted request, an authenticated schema-invalid request, exhaustion of one caller’s allowance, and continued access for the other caller. The malformed authenticated request counts toward the allowance. Keep every supported admitted request real; do not add a fixed answer or mock provider result.

The request plan below is a starting point. Change its order or route to fit your retained project, predict the statuses first, and inspect the real result. It makes at most two supported model calls if you keep one admitted supported request for each caller.


In [ ]:
assessment_settings = settings.replace(
    "POLICY = None",
    "POLICY = LimitPolicy(allowance=2, window_seconds=60)",
)
(PROJECT / "app" / "security_settings.py").write_text(assessment_settings)
service = start_service()

ASSESSMENT_PLAN = [
    {"label": "missing key", "caller": None, "body": payload},
    {"label": "wrong key", "caller": "wrong", "body": payload},
    {"label": "Dispatch admitted", "caller": "dispatch", "body": payload},
    {"label": "Dispatch schema-invalid", "caller": "dispatch", "body": {}},
    {"label": "Dispatch over limit", "caller": "dispatch", "body": payload},
    {"label": "Partner independent allowance", "caller": "partner", "body": payload},
]

assessment_responses = []
with httpx.Client(base_url=service.base_url, timeout=90) as client:
    for step in ASSESSMENT_PLAN:
        headers = {}
        if step["caller"] in caller_keys:
            headers["X-API-Key"] = caller_keys[step["caller"]]
        elif step["caller"] == "wrong":
            headers["X-API-Key"] = "not-a-valid-key"
        response = client.post("/v1/diagnose", json=step["body"], headers=headers)
        assessment_responses.append(response)
        body = response.json()
        print(
            step["label"], "->", response.status_code,
            body.get("status", body.get("detail")),
            "Retry-After:", response.headers.get("retry-after"),
        )
service.stop()


## 7. Save and carry the secured service forward

Record your policy, ordered predictions, actual statuses, and one limitation of the in-memory policy. Keep caller and provider credentials out of notes. Export the app and data, then save the notebook separately in Drive.

In [ ]:
from zipfile import ZIP_DEFLATED, ZipFile

CHECKPOINT_OUT = Path("/content/sprint-2-campus-checkpoint.zip")
with ZipFile(CHECKPOINT_OUT, "w", ZIP_DEFLATED) as archive:
    for folder in ("app", "data"):
        for path in (PROJECT / folder).rglob("*"):
            if path.is_file() and "__pycache__" not in path.parts and path.suffix != ".pyc":
                archive.write(path, path.relative_to(PROJECT))
print("Saved checkpoint:", CHECKPOINT_OUT)

### Sprint 2 checkpoint

You carried forward the Sprint 1 routes and contract, added caller authentication, and limited each caller independently. Sprint 3 adds streaming and safe operational records to this same service.